#Task 8: Multi-Agent Collaborative Swarm with Shared Transactional Blackboard Architecture
###Objective

Architect complex, multi-agent cooperative workflows to solve multi-step analytical problems, utilizing decentralized consensus and shared memory stores.

###Technologies / Tools Used

AutoGen,
Docker-Compose,
PostgreSQL,
Redis,
OpenAI API / Ollama

#Step 1: Install Required Libraries

Install the libraries needed to demonstrate the shared-memory and multi-agent architecture.

In [8]:
# Install required libraries
!pip install -q redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 3.0 MB/s eta 0:00:00


#Step 2: Import Libraries

In [10]:
# Import required libraries
import threading
import time
import sqlite3
import json

#Step 3: Give an Input Prompt

In [11]:
# Define the specialist agents
agents = {
    "Code Generator": "Generates code",
    "System Auditor": "Audits the system",
    "QA Analyst": "Tests the result"
}

# Display the agents
for name, role in agents.items():
    print(name, ":", role)

Code Generator : Generates code
System Auditor : Audits the system
QA Analyst : Tests the result


#Step 4: Create the Shared Blackboard

The Blackboard acts as shared memory. All agents can read information from it and write their results to it.

In [12]:
# Create the shared Blackboard
blackboard = {}

# Store the initial task
blackboard["task"] = "Build and verify a simple system"

print("Shared Blackboard:")
print(blackboard)

Shared Blackboard:
{'task': 'Build and verify a simple system'}


#Step 5: Agent Communication Through the Blackboard

Each specialist performs its responsibility and stores its result in the shared Blackboard.

In [13]:
# Code Generator writes its result
blackboard["code"] = "Code generated successfully."

# System Auditor writes its result
blackboard["audit"] = "System audit completed."

# QA Analyst writes its result
blackboard["qa"] = "QA testing completed."

# Display Blackboard contents
print("Blackboard contents:\n")
print(json.dumps(blackboard, indent=4))

Blackboard contents:

{
    "task": "Build and verify a simple system",
    "code": "Code generated successfully.",
    "audit": "System audit completed.",
    "qa": "QA testing completed."
}


#Step 6: Implement State Locking

A state lock prevents two agents from modifying the same Blackboard state at the same time.

In [15]:
# Create a dictionary to store locks
locks = {}

# Lock a Blackboard key
def lock_key(key):

    if key not in locks:
        locks[key] = True
        return True

    return False


# Unlock a Blackboard key
def unlock_key(key):

    if key in locks:
        del locks[key]

#Step 7: Test the State Lock

The agent first locks the required state, performs its operation, and then releases the lock.

In [16]:
# Try to lock the final output
if lock_key("final_output"):

    print("final_output locked.")

    # Write the final result
    blackboard["final_output"] = "Approved final result."

    # Release the lock
    unlock_key("final_output")

    print("final_output unlocked.")

else:
    print("Could not acquire the lock.")

final_output locked.
final_output unlocked.


#Step 8: Simulate Deadlock Detection and Resolution

A deadlock occurs when agents wait indefinitely for resources locked by each other. Here, we detect the situation and release the locked resources.

In [17]:
# Simulate two locked resources
locks["resource_A"] = "Agent 1"
locks["resource_B"] = "Agent 2"

print("Current locks:")
print(locks)

# Detect possible deadlock
if "resource_A" in locks and "resource_B" in locks:

    print("\nPossible deadlock detected.")

    # Resolve the deadlock by releasing the locks
    del locks["resource_A"]
    del locks["resource_B"]

    print("Deadlock resolved.")

print("\nCurrent locks:")
print(locks)

Current locks:
{'resource_A': 'Agent 1', 'resource_B': 'Agent 2'}

Possible deadlock detected.
Deadlock resolved.

Current locks:
{}


#Step 9: Implement Agent Consensus

The agents read the shared results and reach a decentralized consensus before producing the final output.

Code Cell 9

In [18]:
# Collect the agent results
agent_results = [
    blackboard["code"],
    blackboard["audit"],
    blackboard["qa"]
]

# Check whether all agents completed their work
if all(agent_results):

    blackboard["decision"] = "All agents approved the result."

else:

    blackboard["decision"] = "Further analysis is required."


# Display the final decision
print("Agent Consensus:")
print(blackboard["decision"])

Agent Consensus:
All agents approved the result.


#Step 10: Commit the Final Output to a Transactional Database

A transaction ensures that the final output is stored successfully. We use SQLite in Colab as a lightweight demonstration of the transactional database requirement.

In [19]:
# Create a SQLite database
connection = sqlite3.connect("blackboard.db")

# Create a cursor
cursor = connection.cursor()

# Create the results table
cursor.execute("""
CREATE TABLE IF NOT EXISTS final_results (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    result TEXT
)
""")

# Start a transaction
try:

    # Insert the final result
    cursor.execute(
        "INSERT INTO final_results (result) VALUES (?)",
        (blackboard["decision"],)
    )

    # Commit the transaction
    connection.commit()

    print("Final output committed successfully.")

except Exception as error:

    # Roll back if an error occurs
    connection.rollback()

    print("Transaction failed:", error)

finally:

    connection.close()

Final output committed successfully.


#Step 11: Verify the Transaction

Read the stored result from the database to verify that the final output was successfully committed.

In [20]:
# Connect to the database
connection = sqlite3.connect("blackboard.db")

# Create cursor
cursor = connection.cursor()

# Retrieve stored results
cursor.execute(
    "SELECT * FROM final_results"
)

results = cursor.fetchall()

# Display results
print("Stored Final Results:\n")

for result in results:
    print(result)

# Close connection
connection.close()

Stored Final Results:

(1, 'All agents approved the result.')


#Step 12: Visualize the Multi-Agent Workflow

This visualization shows the flow from specialist agents to the shared Blackboard, consensus, and final transactional storage.

In [21]:
# Display the architecture
print("""
       MULTI-AGENT SWARM

   +-------------------+
   |  Code Generator   |
   +-------------------+
             |
             v
   +-------------------+
   |                   |
   | Shared Blackboard |
   |                   |
   +-------------------+
      ^            ^
      |            |
      |            |
+-------------+ +-------------+
|   System    | |     QA      |
|   Auditor   | |   Analyst   |
+-------------+ +-------------+
             |
             v
   +-------------------+
   | Agent Consensus   |
   +-------------------+
             |
             v
   +-------------------+
   | Transactional DB  |
   +-------------------+
""")


       MULTI-AGENT SWARM

   +-------------------+
   |  Code Generator   |
   +-------------------+
             |
             v
   +-------------------+
   |                   |
   | Shared Blackboard |
   |                   |
   +-------------------+
      ^            ^
      |            |
      |            |
+-------------+ +-------------+
|   System    | |     QA      |
|   Auditor   | |   Analyst   |
+-------------+ +-------------+
             |
             v
   +-------------------+
   | Agent Consensus   |
   +-------------------+
             |
             v
   +-------------------+
   | Transactional DB  |
   +-------------------+



#Conclusion

The multi-agent system demonstrates how independent specialist agents can collaborate through a shared Blackboard. State locking is used to avoid conflicting access, deadlock handling allows the workflow to recover from resource conflicts, and the final consensus is stored in a transactional database